In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import corner
import ultranest

In [ ]:
# Use parser from utils/parse_data.py to build the DataFrame
from utils.parse_data import load_data
raw = load_data()
data = raw['data']
df = pd.DataFrame({
    'N': data['N'],
    'Z': data['Z'],
    'A': data['A'],
    'B': data['BINDING_ENERGY'],
    'B_err': data['BINDING_ENERGY_ERROR']
})
# Clean and compute derived quantities
df = df.dropna(subset=['A','B'])
df = df[df['A']>0].reset_index(drop=True)
df['B_per_A'] = df['B']/df['A']
df['B_per_A_err'] = df['B_err']/df['A']
print('Loaded data from utils/parse_data.load_data(), rows =', len(df))

**Likelihood and systematic uncertainty**

We use a Gaussian likelihood with an extra systematic term: $$\sigma_{\mathrm{sys}}$$ so that:

$$\sigma_i^2 = \sigma_{i,\mathrm{exp}}^2 + \sigma_{\mathrm{sys}}^2.$$

The log-likelihood becomes:

$$\log L = -\tfrac{1}{2}\sum_i\left[\frac{(B_i^{\mathrm{obs}}-B_i^{\mathrm{mod}})^2}{\sigma_i^2} + \ln(2\pi\sigma_i^2)\right]$$

In [ ]:
# Bethe-Weizsäcker model WITHOUT pairing term, with prior and likelihood (uses same sigma_sys treatment)
def bethe_weizsacker_no_pairing(A, Z, a_v, a_s, a_c, a_sym):
    A = np.array(A, dtype=float)
    Z = np.array(Z, dtype=float)
    B = a_v*A - a_s*A**(2.0/3.0) - a_c*Z*(Z-1)/A**(1.0/3.0) - a_sym*(A-2*Z)**2/A
    return B

# Prior transform for no-pairing model: 5 parameters (a_v,a_s,a_c,a_sym,sigma_sys)
def prior_transform_nopair(u):
    a_v = 5.0 + u[0]*35.0      # 5-40 MeV
    a_s = 0.0 + u[1]*50.0      # 0-50 MeV
    a_c = 0.0 + u[2]*1.5       # 0-1.5 MeV
    a_sym = 0.0 + u[3]*60.0    # 0-60 MeV
    log10_sigma = -3.0 + u[4]*4.0  # sigma_sys from 1e-3 to 10 (same units as B)
    sigma_sys = 10**log10_sigma
    return [a_v, a_s, a_c, a_sym, sigma_sys]

def log_likelihood_nopair(params):
    # params: a_v,a_s,a_c,a_sym,sigma_sys
    a_v,a_s,a_c,a_sym,sigma_sys = params
    B_model = bethe_weizsacker_no_pairing(A_arr, Z_arr, a_v,a_s,a_c,a_sym)
    sigma2 = B_err**2 + sigma_sys**2
    ll = -0.5*np.sum((B_obs - B_model)**2/sigma2 + np.log(2*np.pi*sigma2))
    return ll

print('Defined no-pairing model, prior_transform_nopair and log_likelihood_nopair')